# Evaluate anchored registry strategies

**Feature:** NB-006  
**Goal:** compare top-1 and top-2 counterfactual quality, synchronized runtime, module-state footprint, and exact active-MLP work through public APIs.

This notebook is a deterministic offline smoke workflow. Wall-clock values describe only the current process and hardware; route-derived work counts are exact.

## Resolve configuration

One mapping declares the seed, registry order, token shape, warmup, repetitions, work-unit definition, and device.

In [ ]:
import json
import os

import torch
from torch import nn

from anchored_moe import (
    AffineRouterHead,
    AnchorVetoTop2Selection,
    AnchoredTop1Dispatcher,
    AnchoredTop1Selection,
    AnchoredTop2Dispatcher,
    AppendOnlyRouterHeads,
    CounterfactualRoute,
    CounterfactualRouteLosses,
    EfficiencyBenchmarkConfig,
    HierarchicalPackageTop1LocalTop2Selection,
    HierarchicalRouterScoreBank,
    benchmark_operation,
    compare_top1_top2_to_oracle,
    compile_logical_router,
    measure_active_mlp_metrics,
)

SMOKE_TEST = os.environ.get("ANCHORED_MOE_SMOKE_TEST", "1") == "1"
EXPERIMENT = {
    "feature_id": "NB-006",
    "seed": 13,
    "expert_ids": ("code", "math"),
    "hierarchical_package_ids": ("code-package", "math-package"),
    "token_shape": (2, 4),
    "hidden_size": 4,
    "warmup_runs": 1,
    "measured_runs": 2 if SMOKE_TEST else 20,
    "device": "cpu",
    "work_unit": "tokens",
}
EXPERIMENT

## Prepare the deterministic registry fixture

The same anchor, experts, hidden states, calibrated scores, and registry order are used by both strategies. Production evaluation should replace only these fixture tensors and modules.

In [ ]:
torch.manual_seed(EXPERIMENT["seed"])
hidden_states = torch.randn(*EXPERIMENT["token_shape"], EXPERIMENT["hidden_size"])
expert_scores = torch.tensor([
    [[-1.0, -2.0], [2.0, 1.0], [0.5, 1.5], [1.5, 1.0]],
    [[-0.5, -0.1], [0.8, 0.7], [1.2, 0.9], [0.2, -0.4]],
])
anchor = nn.Linear(EXPERIMENT["hidden_size"], EXPERIMENT["hidden_size"]).eval()
code = nn.Linear(EXPERIMENT["hidden_size"], EXPERIMENT["hidden_size"]).eval()
math = nn.Linear(EXPERIMENT["hidden_size"], EXPERIMENT["hidden_size"]).eval()
modules = (anchor, code, math)
for module in modules:
    module.requires_grad_(False)
trainable_parameter_count = sum(parameter.numel() for module in modules for parameter in module.parameters() if parameter.requires_grad)
assert trainable_parameter_count == 0
top1_dispatcher = AnchoredTop1Dispatcher()
top2_dispatcher = AnchoredTop2Dispatcher()
top1_operation = lambda: top1_dispatcher.dispatch(
    hidden_states,
    expert_scores=expert_scores,
    expert_ids=EXPERIMENT["expert_ids"],
    anchor_mlp=anchor,
    experts=(code, math),
)
top2_operation = lambda: top2_dispatcher.dispatch(
    hidden_states,
    expert_scores=expert_scores,
    expert_ids=EXPERIMENT["expert_ids"],
    anchor_mlp=anchor,
    experts=(code, math),
)
package_heads = AppendOnlyRouterHeads(feature_size=2)
code_local_heads = AppendOnlyRouterHeads(feature_size=2)
math_local_heads = AppendOnlyRouterHeads(feature_size=2)
for identifier, weight in (("code-package", (1.0, 0.0)), ("math-package", (0.0, 1.0))):
    head = AffineRouterHead(2, layer_index=0, expert_id=identifier, trainable=False)
    with torch.no_grad():
        head.linear.weight.copy_(torch.tensor([weight]))
    package_heads.append(head)
for bank, rows in (
    (code_local_heads, (("code:a", (1.0, 0.0)), ("code:b", (0.0, 1.0)))),
    (math_local_heads, (("math:a", (-1.0, 0.0)), ("math:b", (0.0, -1.0)))),
):
    for identifier, weight in rows:
        head = AffineRouterHead(2, layer_index=0, expert_id=identifier, trainable=False)
        with torch.no_grad():
            head.linear.weight.copy_(torch.tensor([weight]))
        bank.append(head)
hierarchical_router = HierarchicalRouterScoreBank(
    compile_logical_router(package_heads),
    {
        "code-package": compile_logical_router(code_local_heads),
        "math-package": compile_logical_router(math_local_heads),
    },
)
hierarchical_policy = HierarchicalPackageTop1LocalTop2Selection()

## Run framework operations

EVAL-004 consumes explicit counterfactual losses. EVAL-005 benchmarks the two sparse dispatch closures with identical work units and separately derives exact MLP token executions.

In [ ]:
counterfactual_losses = CounterfactualRouteLosses(
    torch.tensor([
        [0.5, 0.9, 1.0, 0.8], [1.0, 0.8, 0.9, 0.2],
        [1.0, 0.9, 0.8, 0.2], [1.0, 0.4, 0.6, 0.7],
        [0.4, 0.8, 0.7, 0.6], [1.0, 0.6, 0.8, 0.4],
        [1.0, 0.5, 0.7, 0.3], [0.8, 0.6, 0.9, 0.7],
    ]).reshape(*EXPERIMENT["token_shape"], 4),
    expert_ids=EXPERIMENT["expert_ids"],
    routes=(
        CounterfactualRoute.anchor(),
        CounterfactualRoute.single("code"),
        CounterfactualRoute.single("math"),
        CounterfactualRoute.pair("code", "math"),
    ),
    loss_name="cross_entropy",
)
top1_decision = AnchoredTop1Selection()(expert_scores, expert_ids=EXPERIMENT["expert_ids"])
top2_decision = AnchorVetoTop2Selection()(expert_scores, expert_ids=EXPERIMENT["expert_ids"])
quality = compare_top1_top2_to_oracle(counterfactual_losses, top1_decision, top2_decision)
benchmark_config = EfficiencyBenchmarkConfig(
    warmup_runs=EXPERIMENT["warmup_runs"],
    measured_runs=EXPERIMENT["measured_runs"],
    work_items_per_run=hidden_states[..., 0].numel(),
    device=EXPERIMENT["device"],
)
with torch.no_grad():
    top1_result = top1_operation()
    top2_result = top2_operation()
    top1_benchmark = benchmark_operation(top1_operation, config=benchmark_config, modules=modules)
    top2_benchmark = benchmark_operation(top2_operation, config=benchmark_config, modules=modules)
top1_work = measure_active_mlp_metrics(top1_result.selected_routes)
top2_work = measure_active_mlp_metrics(top2_result.selected_routes)
hierarchical_features = hidden_states[..., :2]
hierarchical_package_routes = hierarchical_policy.select_packages(
    hierarchical_router.score_packages(hierarchical_features, layer_index=0),
    package_ids=hierarchical_router.package_ids_for_layer(0),
)
hierarchical_local_scores = hierarchical_router.score_selected_locals(
    hierarchical_features,
    package_indices=hierarchical_package_routes.selected_indices,
    package_ids=hierarchical_package_routes.expert_ids,
    layer_index=0,
)
hierarchical_routes = hierarchical_policy.select_locals(
    hierarchical_package_routes,
    hierarchical_local_scores,
    expert_ids_by_package={
        package_id: hierarchical_router.expert_ids_for_package(package_id, layer_index=0)
        for package_id in hierarchical_router.package_ids
    },
)

## Validate invariants and summarize

Both strategies must use identical resident state, top-1 must execute exactly one MLP per token, top-2 work must match its approved-pair count, and every runtime sample must be positive. Timing is reported without a machine-specific pass threshold.

In [ ]:
assert top1_benchmark.module_state == top2_benchmark.module_state
assert top1_benchmark.module_state is not None
assert top1_benchmark.module_state.trainable_parameter_bytes == 0
assert top1_benchmark.peak_cuda_memory_bytes is None
assert top2_benchmark.peak_cuda_memory_bytes is None
assert top1_work.mean_active_mlps_per_token == 1.0
assert top2_work.total_mlp_token_executions == 13
assert top2_work.mean_active_mlps_per_token == 13 / 8
assert top1_benchmark.measured_runs == EXPERIMENT["measured_runs"]
assert top2_benchmark.measured_runs == EXPERIMENT["measured_runs"]
assert quality.mean_top2_regret < quality.mean_top1_regret
assert torch.equal(hierarchical_routes.anchor_mask, hierarchical_package_routes.anchor_mask)
assert sum(group.expert_scores.shape[0] for group in hierarchical_local_scores) == int((~hierarchical_routes.anchor_mask).sum().item())
for token_package, token_pair in zip(
    hierarchical_package_routes.selected_indices.reshape(-1).tolist(),
    hierarchical_routes.expert_routes.selected_indices.reshape(-1, 2).tolist(),
    strict=True,
):
    if token_package == 0:
        assert token_pair == [0, 0]
    else:
        assert {hierarchical_routes.expert_package_ids[index - 1] for index in token_pair} == {hierarchical_package_routes.expert_ids[token_package - 1]}
evaluation_summary = {
    "feature_id": EXPERIMENT["feature_id"],
    "trainable_parameter_count": trainable_parameter_count,
    "work_unit": EXPERIMENT["work_unit"],
    "quality": {
        "mean_top1_regret": quality.mean_top1_regret,
        "mean_top2_regret": quality.mean_top2_regret,
    },
    "top1": {"runtime": top1_benchmark.to_dict(), "work": top1_work.to_dict()},
    "top2": {"runtime": top2_benchmark.to_dict(), "work": top2_work.to_dict()},
    "hierarchical": {
        "package_ids": list(hierarchical_router.package_ids),
        "anchor_tokens": int(hierarchical_routes.anchor_mask.sum().item()),
        "locally_scored_tokens": sum(group.expert_scores.shape[0] for group in hierarchical_local_scores),
    },
}
print("NB-006 validation passed: " + json.dumps(evaluation_summary, sort_keys=True))